# ASR: Demo notebook

This project provides tools for training and evaluating automatic speech
recognition (ASR) models. It includes implementations based on
[Conformer](https://arxiv.org/abs/2005.08100) and
[FastConformer](https://arxiv.org/abs/2305.05084), trained with CTC loss.

The main FastConformer model is trained on LibriSpeech (960 hours).
Text is represented using a SentencePiece BPE tokenizer with a vocabulary
of 256 tokens.

This notebook compares greedy decoding and beam search with a KenLM 4-gram
language model. It shows how to set up the environment, download the weights,
transcribe a small example, and process your own dataset from a public link.

Predictions are saved as separate text files with the original utterance IDs.
When reference transcripts are available, CER and WER are calculated and
averaged over utterances. An optional section evaluates the model on the
LibriSpeech dev and test splits.

[GitHub](https://github.com/k1ngofdarks/Automatic-Speech-Recognition)
· [Model weights and KenLM](https://huggingface.co/k1ngofdarks/dla-asr)
· [Experiment report](https://www.comet.com/alexander-stepanenko/dla-asr/reports/T2Me7FLe16PcCB7Bvgnb0mz66)



## 1. Environment setup

This notebook is intended for a fresh Google Colab session with a GPU.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/Automatic-Speech-Recognition") # For a local run, set this to your project directory.

if not PROJECT.exists():
    !git clone https://github.com/k1ngofdarks/Automatic-Speech-Recognition.git {PROJECT}

%cd {PROJECT}

### Requirements


In [ ]:
%pip install -r requirements.txt

### Imports


In [ ]:
import torch
import hydra
import soundfile as sf
import pandas as pd
import json

### Configuration


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
MODEL_PATH = PROJECT / "saved/final/model.pth"
LM_PATH = PROJECT / "data/kenlm/4-gram.arpa"
TRAINING_LOG_PATH = PROJECT / "saved/final/info.log"
# These paths match src/configs/download_resources.yaml.

In [ ]:
OUTPUT_ROOT = PROJECT / "demo_output"

## 2. Download resources

The model weights, language model, and complete training log are hosted on
[Hugging Face](https://huggingface.co/k1ngofdarks/dla-asr).

`scripts/download_resources.py` downloads them using `huggingface_hub`.
File names and local directories are defined in
`src/configs/download_resources.yaml`. Existing files are skipped.

Downloaded resources:

- `saved/final/model.pth`: the final model;
- `data/kenlm/4-gram.arpa`: an external LibriSpeech 4-gram language model
  in ARPA format, also available on [OpenSLR](https://www.openslr.org/11/).
- `saved/final/info.log`: the complete final-model training log for epochs 1–30.

The BPE tokenizer at `data/tokenizers/train-all/bpe-256/tokenizer.model`
is already included in the GitHub repository. It was trained on transcripts
from the LibriSpeech training splits. Tokenizer training instructions are
available in the README.

The final model was trained for 30 full epochs using the `fc_big_run` config.
Training and resume commands are available in the README.

The cell below runs the download script from the project directory and checks
that all required resources are available.


In [ ]:
%run scripts/download_resources.py

TOKENIZER_PATH = PROJECT / "data/tokenizers/train-all/bpe-256/tokenizer.model"

for resource_path in (MODEL_PATH, LM_PATH, TOKENIZER_PATH, TRAINING_LOG_PATH):
    if not resource_path.is_file():
        raise FileNotFoundError(resource_path)
    size_mib = resource_path.stat().st_size / 1024**2
    print(f"{resource_path}: {size_mib:.1f} MiB")


## 3. Small example

We use two short English recordings with reference transcripts.
The files are downloaded from a Google Drive folder and transcribed with
two decoders: greedy and beam search with KenLM 4-gram.

After inference, you can listen to each recording and compare its reference
with both predictions in a table. Results are saved to `demo_output/example/`.
We then run `scripts/calc_metrics.py` separately for each decoder.

The table shows CER and WER as percentages, averaged over utterances.
They should match the metrics printed by inference after conversion to
percentages. These two examples demonstrate the workflow; evaluation on
the full dev and test splits is covered in a separate section.


In [ ]:
import gdown

SAMPLE_URL = "https://drive.google.com/drive/folders/1wN4q3Q5pq44flWdJKGYyiIeqJXgc-W3U"
SAMPLE_DIR = PROJECT / "data/datasets/demo_samples"

gdown.download_folder(SAMPLE_URL, output=str(SAMPLE_DIR))


In [ ]:
SAMPLE_OUTPUT = OUTPUT_ROOT / "example"

%run inference.py -cn=inference_customdirdataset \
    custom_dir={SAMPLE_DIR} \
    inferencer.from_pretrained={MODEL_PATH} \
    inferencer.device={DEVICE} \
    inferencer.save_path={SAMPLE_OUTPUT}


In [ ]:
from IPython.display import Audio, display

prediction_rows = []

for audio_path in sorted((SAMPLE_DIR / "audio").glob("*.flac")):
    id = audio_path.stem
    print(id)
    audio, sample_rate = sf.read(audio_path)
    display(Audio(audio, rate=sample_rate, normalize=False))

    target_path = SAMPLE_DIR / "transcriptions" / f"{id}.txt"
    greedy_path = SAMPLE_OUTPUT / "customdataset" / "greedy" / f"{id}.txt"
    beam_lm_path = SAMPLE_OUTPUT / "customdataset" / "beam_lm" / f"{id}.txt"

    prediction_rows.append({
        "ID": id,
        "Reference": target_path.read_text().strip(),
        "Greedy": greedy_path.read_text().strip(),
        "Beam + LM": beam_lm_path.read_text().strip(),
    })

predictions_df = pd.DataFrame(prediction_rows).set_index("ID")
display(predictions_df.style.set_properties(**{
    "white-space": "normal",
    "text-align": "left",
}))


Calculate metrics


In [ ]:
TARGET_DIR = SAMPLE_DIR / "transcriptions"
metric_rows = []

for decoder_key, decoder_name in (("greedy", "Greedy"), ("beam_lm", "Beam + LM")):
    PREDICTION_DIR = SAMPLE_OUTPUT / "customdataset" / decoder_key
    METRICS_PATH = SAMPLE_OUTPUT / f"{decoder_key}_metrics.json"

    %run -m scripts.calc_metrics target_dir={TARGET_DIR} prediction_dir={PREDICTION_DIR} output_path={METRICS_PATH} 

    scores = json.loads(METRICS_PATH.read_text())
    metric_rows.append({"Decoder": decoder_name, "CER": 100 * scores["CER"], "WER": 100 * scores["WER"]})

metrics_df = pd.DataFrame(metric_rows).set_index("Decoder")
display(metrics_df.style.format({"CER": "{:.2f}", "WER": "{:.2f}"}))


## 4. Transcribe your own dataset

Change only `DATASET_URL` to a public Google Drive folder link.
The default link points to the small example from section 3.

The folder must have the following structure:

```text
dataset/
├── audio/
│   ├── utterance1.wav
│   ├── utterance2.flac
│   └── ...
└── transcriptions/       # Optional.
    ├── utterance1.txt
    ├── utterance2.txt
    └── ...
```

Supported extensions are `.wav`, `.flac`, and `.mp3`. Reference `.txt` files
must have the same IDs as the audio files. If `transcriptions/` is absent,
only predictions are saved, and CER/WER are not calculated.


In [ ]:
# Public Google Drive folder containing your dataset.
DATASET_URL = "https://drive.google.com/drive/folders/1wN4q3Q5pq44flWdJKGYyiIeqJXgc-W3U"


In [ ]:
import gdown

USER_DATASET_DIR = PROJECT / "data/datasets/custom_dataset"
gdown.download_folder(DATASET_URL, output=str(USER_DATASET_DIR))

USER_AUDIO_FILES = sorted(
    path for path in (USER_DATASET_DIR / "audio").iterdir()
    if path.suffix in {".wav", ".flac", ".mp3"}
)
USER_TARGET_DIR = USER_DATASET_DIR / "transcriptions"
HAS_TRANSCRIPTIONS = USER_TARGET_DIR.is_dir()

print("Dataset:", USER_DATASET_DIR)
print("Audio files:", len(USER_AUDIO_FILES))


In [ ]:
USER_OUTPUT = OUTPUT_ROOT / "custom_dataset"

%run inference.py -cn=inference_customdirdataset \
    custom_dir={USER_DATASET_DIR} \
    inferencer.from_pretrained={MODEL_PATH} \
    inferencer.device={DEVICE} \
    inferencer.save_path={USER_OUTPUT}


In [ ]:
if HAS_TRANSCRIPTIONS:
    user_metric_rows = []

    for decoder_key, decoder_name in (("greedy", "Greedy"), ("beam_lm", "Beam + LM")):
        PREDICTION_DIR = USER_OUTPUT / "customdataset" / decoder_key
        METRICS_PATH = USER_OUTPUT / f"{decoder_key}_metrics.json"


        %run -m scripts.calc_metrics target_dir={USER_TARGET_DIR} prediction_dir={PREDICTION_DIR} output_path={METRICS_PATH}

        scores = json.loads(METRICS_PATH.read_text())
        user_metric_rows.append({"Decoder": decoder_name, "CER": 100 * scores["CER"], "WER": 100 * scores["WER"]})

    user_metrics_df = pd.DataFrame(user_metric_rows).set_index("Decoder")
    display(user_metrics_df.style.format({"CER": "{:.2f}", "WER": "{:.2f}"}))
else:
    print("Transcriptions not found: CER/WER metrics will not be calculated.")


In [ ]:
from IPython.display import display

user_prediction_rows = []
for audio_path in USER_AUDIO_FILES[:5]:
    id = audio_path.stem
    row = {"ID": id}
    if HAS_TRANSCRIPTIONS:
        row["Reference"] = (USER_TARGET_DIR / f"{id}.txt").read_text(encoding="utf-8").strip()

    for decoder_key, decoder_name in (("greedy", "Greedy"), ("beam_lm", "Beam + LM")):
        prediction_path = USER_OUTPUT / "customdataset" / decoder_key / f"{id}.txt"
        row[decoder_name] = prediction_path.read_text(encoding="utf-8").strip()
    user_prediction_rows.append(row)

user_predictions_df = pd.DataFrame(user_prediction_rows).set_index("ID")
display(user_predictions_df)
print("Full results:", USER_OUTPUT)


## 5. LibriSpeech

This section is optional. It downloads four LibriSpeech splits if needed
and evaluates greedy and beam search with KenLM on each split.
This takes more time and disk space, so `RUN_LIBRISPEECH = False` by default.

To run the full evaluation, set `RUN_LIBRISPEECH = True` and execute the cell.
The `datasets=all_eval` config includes `dev_clean`, `dev_other`, `test_clean`,
and `test_other`. The weights, tokenizer, and LM settings are the same as
in the previous sections.

The script prints CER/WER separately for each split and decoder.
Predictions are saved to
`demo_output/librispeech/<split>/<decoder>/<utterance_id>.txt`.


In [ ]:
RUN_LIBRISPEECH = False 
# If True, run full evaluation on LibriSpeech dataset. 
# Work ~ 1 hour (I don't know why colab so slow download LibriSpeech dataset)

LIBRISPEECH_OUTPUT = OUTPUT_ROOT / "librispeech"


if RUN_LIBRISPEECH:
    %run inference.py -cn=inference_fastconformer datasets=all_eval \
        inferencer.from_pretrained={MODEL_PATH} \
        inferencer.device={DEVICE} \
        inferencer.save_path={LIBRISPEECH_OUTPUT}
else:
    print("Full evaluation of LibriSpeech skipped. To run, set RUN_LIBRISPEECH = True.")


### Final model results

Results of the final checkpoint `model.pth` from epoch 30 on all four splits.
CER and WER are averaged over utterances and reported as percentages.

Beam + LM uses KenLM 4-gram with `alpha=0.5` and `beta=1.0`.
These are the default settings used for the results in the [experiment report](https://www.comet.com/alexander-stepanenko/dla-asr/reports/T2Me7FLe16PcCB7Bvgnb0mz66).

| Split | Greedy CER | Greedy WER | Beam + LM CER | Beam + LM WER |
| --- | --- | --- | --- | --- |
| dev-clean | 3.09 | 9.06 | 2.33 | 6.12 |
| test-clean | 3.01 | 8.82 | 2.29 | 6.09 |
| dev-other | 8.76 | 20.50 | 6.84 | 14.04 |
| test-other | 8.87 | 21.09 | 7.13 | 14.90 |


## 6. Download results

Create a ZIP archive with the results of the custom dataset run from section 4.
It contains text predictions and, when references are available, JSON files
with metrics.

```text
customdataset/
├── greedy/<utterance_id>.txt
└── beam_lm/<utterance_id>.txt
greedy_metrics.json          # Only when references are available.
beam_lm_metrics.json         # Only when references are available.
```

In Colab, the cell starts downloading the archive. When running locally,
it prints the path to the created file.


In [ ]:
import shutil

RESULTS_ZIP = shutil.make_archive(str(USER_OUTPUT), "zip", root_dir=USER_OUTPUT)
print("Results archive:", RESULTS_ZIP)

try:
    from google.colab import files
except ImportError:
    print("When running locally, open the archive at the path shown above.")
else:
    files.download(RESULTS_ZIP)


## 7. Training and reproducibility

The model is trained from scratch on three LibriSpeech training splits:
`train-clean-100`, `train-clean-360`, and `train-other-500`, totaling 960 hours
before dataset filtering. Training completed 30 full epochs; the published
`model.pth` is the checkpoint from epoch 30. The main training config is
[`src/configs/fc_big_run.yaml`](src/configs/fc_big_run.yaml).

| Parameter | Value |
| --- | --- |
| Model | Custom FastConformer + CTC implementation, about 8.5 million parameters |
| Encoder | 16 blocks, dimension 144, 4 attention heads, FFN dimension 576 |
| Subsampling | 8x along time, 256 channels in the convolutional frontend |
| Features | 128 mel bins, log transformation and normalization |
| Tokenizer | SentencePiece BPE, 256 tokens + CTC blank |
| Training augmentations | Two frequency masks of up to 16 bins and two time masks of up to 50 frames |
| Optimizer | AdamW, max LR `3e-4` |
| Scheduler | OneCycleLR over 30 full epochs, 5% warmup |
| Batch size | 16 |
| Hardware and precision | RTX 4060 8 GB, AMP |
| Saving | A checkpoint after every epoch |

Training command from the project root (for reference; training is not run
in this demo):

```bash
python train.py -cn=fc_big_run writer.run_name=fastconformer_own_t960_bs1
```

Resume with the saved config, for example after epoch 20:

```bash
python train.py \
  --config-path saved/fastconformer_own_t960_bs1 \
  --config-name config \
  trainer.resume_from=checkpoint-epoch20.pth
```

The checkpoint includes model weights, optimizer, scheduler, and AMP scaler
states. Training resumes from the next epoch.

The complete training log for epochs 1–30 is downloaded to
`saved/final/info.log` in section 2.

[Training metrics on Comet](https://www.comet.com/alexander-stepanenko/dla-asr/e9ltoobnfmw1t4yob6xsdpaw121nxcai).
[Experiment report with training curves, comparisons, and final results](https://www.comet.com/alexander-stepanenko/dla-asr/reports/T2Me7FLe16PcCB7Bvgnb0mz66).
General project commands are available in the [README](README.md).
